In [1]:
import numpy as np

In [2]:
# extract audio features using librosa
def extract_features(audio_file):
  import librosa

  y, sr = librosa.load(audio_file, sr=None)
  mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
  chroma = librosa.feature.chroma_stft(y=y, sr=sr)
  spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
  rms = librosa.feature.rms(y=y)[0]

  features = {
      'mfcc': np.mean(mfcc, axis=1),
      'chroma': np.mean(chroma, axis=1),
      'spectral_centroid': np.mean(spectral_centroid),
      'rms': np.mean(rms)
  }

  return features

In [ ]:
!unzip train.zip

In [4]:
class_labels = ['guitar', 'piano', 'strings']
num_classes = len(class_labels)

In [5]:
# build dataset from audio files under train for each instrument

input_x = []
output_y = []

import os

for instrument in class_labels:
  dir_path = os.path.join("train", instrument)
  for filename in os.listdir(dir_path):
    audio_file = os.path.join(dir_path, filename)
    if os.path.isfile(audio_file):

      # extract audio features
      features = extract_features(audio_file)
      feature_vector = np.concatenate([features['mfcc'], features['chroma'], [features['spectral_centroid']], [features['rms']]])
      input_x.append(feature_vector)
      output_y.append(instrument)

input_x = np.array(input_x)
output_y = np.array(output_y)

In [6]:
input_x.shape

(198, 27)

In [7]:
output_y.shape

(198,)

In [8]:
# split data for training and testing

from sklearn.model_selection import train_test_split
train_x, test_x, train_y, test_y = train_test_split(input_x, output_y, shuffle=True, test_size=0.2)

In [9]:
# choose and run ML model

from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier().fit(train_x, train_y)
model.score(test_x, test_y)

0.9

In [10]:
# define neural network architecture

import keras
from keras import layers

inputs = keras.Input(shape=(27, ))
dense1 = layers.Dense(32, activation="relu")(inputs)
dense2 = layers.Dense(32, activation="relu")(dense1)

outputs = layers.Dense(num_classes, activation="softmax")(dense2)
model = keras.Model(inputs=inputs, outputs=outputs)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 27)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,051 (8.01 KB)

 Trainable params: 2,051 (8.01 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

In [12]:
# convert class labels from string to number

from sklearn.preprocessing import LabelEncoder
z = LabelEncoder().fit_transform(output_y)

In [13]:
model.fit(input_x, z, validation_split=0.2, epochs=30, shuffle=True)

Epoch 1/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 73ms/step - accuracy: 0.1020 - loss: 107.2204 - val_accuracy: 0.0000e+00 - val_loss: 84.1331
Epoch 2/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.1185 - loss: 46.4897 - val_accuracy: 0.0000e+00 - val_loss: 24.9867
Epoch 3/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.4314 - loss: 6.5052 - val_accuracy: 0.9750 - val_loss: 0.1928
Epoch 4/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.8561 - loss: 5.4254 - val_accuracy: 0.9750 - val_loss: 0.1330
Epoch 5/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.8748 - loss: 6.5828 - val_accuracy: 0.9750 - val_loss: 0.1454
Epoch 6/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.8505 - loss: 5.1801 - val_accuracy: 0.9750 - val_loss: 0.0515
Epoch 7/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.8214 - loss: 6.9066 - val_accuracy: 0.9750 - val_loss: 0.0528
Epoch 8/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.8392 - loss: 4.5676 - val_accuracy: 0.9750 - val_